# Лаборатори 5: Пропозицийн логик
Үнэний хүснэгт, CNF, резолюц ба DPLL хэрэгжүүлэлт

Оюутан: Jamiyasnuren (s22c156b)

## Алхам 1: Өгүүлбэрийг илэрхийлэх

In [1]:
import itertools, random

def Sym(n):    return ('sym', n)
def Not(a):    return ('not', a)
def And(a, b): return ('and', a, b)
def Or(a, b):  return ('or', a, b)
def Imp(a, b): return ('imp', a, b)
def Iff(a, b): return ('iff', a, b)

def symbols(s):
    """Өгүүлбэрт орсон тэмдэгтүүдийн нэрийн олонлог (рекурсив)."""
    if s[0] == 'sym':
        return {s[1]}
    result = set()
    for sub in s[1:]:
        result |= symbols(sub)
    return result

SYMBOL = {'and': '&', 'or': '|', 'imp': '=>', 'iff': '<=>'}

def to_str(s):
    """Хаалттай, уншигдахуйц мөр, жишээ нь ((P & Q) => R)."""
    if s[0] == 'sym':
        return s[1]
    if s[0] == 'not':
        return '~' + to_str(s[1])
    return '(' + to_str(s[1]) + ' ' + SYMBOL[s[0]] + ' ' + to_str(s[2]) + ')'

In [2]:
# Шалгах (Алхам 1)
P, Q, R = Sym('P'), Sym('Q'), Sym('R')
s1 = Imp(And(P, Q), R)
print(to_str(s1), symbols(s1))
assert symbols(s1) == {'P', 'Q', 'R'} and to_str(s1) == '((P & Q) => R)'
s2 = Iff(Or(P, Not(Q)), Not(R));  print(to_str(s2), sorted(symbols(s2)))
s3 = Not(And(Imp(P, Q), Not(R))); print(to_str(s3), sorted(symbols(s3)))

((P & Q) => R) {'R', 'P', 'Q'}
((P | ~Q) <=> ~R) ['P', 'Q', 'R']
~((P => Q) & ~R) ['P', 'Q', 'R']


## Алхам 2: Семантик

In [3]:
def pl_true(s, model):
    op = s[0]
    if op == 'sym': return model[s[1]]
    if op == 'not': return not pl_true(s[1], model)
    a = pl_true(s[1], model)
    b = pl_true(s[2], model)
    if op == 'and': return a and b
    if op == 'or':  return a or b
    if op == 'imp': return (not a) or b
    if op == 'iff': return a == b
    raise ValueError(op)

In [4]:
# Шалгах (Алхам 2)
m1 = {'P': True, 'Q': False, 'R': True}
print(pl_true(Imp(And(P, Q), R), m1), pl_true(Iff(P, Q), m1), pl_true(Or(Not(P), Q), m1))
assert pl_true(Imp(And(P, Q), R), m1) is True
assert pl_true(Iff(P, Q), m1) is False
assert pl_true(Or(Not(P), Q), m1) is False

True False False


## Алхам 3: Үнэний хүснэгт ба логик дүгнэлт

In [5]:
def all_models(syms):
    """n тэмдэгтийн 2^n загварыг нэг нэгээр нь буцаах generator."""
    syms = sorted(syms)
    for values in itertools.product([True, False], repeat=len(syms)):
        yield dict(zip(syms, values))

def conj(sentences):
    """Олон өгүүлбэрийг And-аар нэг өгүүлбэр болгоно."""
    result = sentences[0]
    for s in sentences[1:]:
        result = And(result, s)
    return result

def tt_entails(kb, alpha):
    """KB |= alpha эсэх. (үр дүн, шалгасан загварын тоо)."""
    if isinstance(kb, list): kb = conj(kb)
    syms = symbols(kb) | symbols(alpha)
    checked = 0
    for model in all_models(syms):
        checked += 1
        if pl_true(kb, model) and not pl_true(alpha, model):
            return False, checked          # эсрэг жишээ олдлоо
    return True, checked

def is_valid(s):
    return all(pl_true(s, m) for m in all_models(symbols(s)))

def is_satisfiable(s):
    return any(pl_true(s, m) for m in all_models(symbols(s)))

def print_truth_table(s):
    syms = sorted(symbols(s))
    print(' '.join(f'{x:>5}' for x in syms) + ' | ' + to_str(s))
    for m in all_models(syms):
        row = ' '.join(f'{str(m[x])[0]:>5}' for x in syms)
        print(row + ' | ' + str(pl_true(s, m))[0])

In [6]:
# Шалгах (Алхам 3)
assert tt_entails(And(P, Imp(P, Q)), Q)[0] is True             # Modus Ponens
assert tt_entails(Or(P, Q), P)[0] is False
assert tt_entails(And(Not(Q), Imp(P, Q)), Not(P))[0] is True   # Modus Tollens
assert is_valid(Or(P, Not(P))) and not is_satisfiable(And(P, Not(P)))
res, cnt = tt_entails(And(P, Imp(P, Q)), Q)
assert cnt == 2 ** 2
print('Modus Ponens:', res, 'шалгасан загвар:', cnt)
# Де Морганы хууль
a, b = Not(And(P, Q)), Or(Not(P), Not(Q))
print_truth_table(a); print_truth_table(b)
assert all(pl_true(a, m) == pl_true(b, m) for m in all_models({'P', 'Q'}))

Modus Ponens: True шалгасан загвар: 4
    P     Q | ~(P & Q)
    T     T | F
    T     F | T
    F     T | T
    F     F | T
    P     Q | (~P | ~Q)
    T     T | F
    T     F | T
    F     T | T
    F     F | T


## Алхам 4: CNF-д хөрвүүлэх

In [7]:
def eliminate_iff(s):
    if s[0] == 'sym': return s
    if s[0] == 'iff':
        a, b = eliminate_iff(s[1]), eliminate_iff(s[2])
        return And(Imp(a, b), Imp(b, a))
    return (s[0],) + tuple(eliminate_iff(x) for x in s[1:])

def eliminate_imp(s):
    if s[0] == 'sym': return s
    if s[0] == 'imp':
        return Or(Not(eliminate_imp(s[1])), eliminate_imp(s[2]))
    return (s[0],) + tuple(eliminate_imp(x) for x in s[1:])

def push_not(s):
    if s[0] == 'sym': return s
    if s[0] == 'not':
        t = s[1]
        if t[0] == 'sym': return s
        if t[0] == 'not': return push_not(t[1])                       # давхар үгүйсгэл
        if t[0] == 'and': return Or(push_not(Not(t[1])), push_not(Not(t[2])))   # Де Морган
        if t[0] == 'or':  return And(push_not(Not(t[1])), push_not(Not(t[2])))
    return (s[0],) + tuple(push_not(x) for x in s[1:])

def distribute(s):
    if s[0] == 'and':
        return And(distribute(s[1]), distribute(s[2]))
    if s[0] == 'or':
        a, b = distribute(s[1]), distribute(s[2])
        if a[0] == 'and':
            return And(distribute(Or(a[1], b)), distribute(Or(a[2], b)))
        if b[0] == 'and':
            return And(distribute(Or(a, b[1])), distribute(Or(a, b[2])))
        return Or(a, b)
    return s

def to_cnf(s):
    return distribute(push_not(eliminate_imp(eliminate_iff(s))))

def neg(lit):
    return lit[1:] if lit.startswith('~') else '~' + lit

def _literals(s):
    if s[0] == 'or':  return _literals(s[1]) + _literals(s[2])
    if s[0] == 'sym': return [s[1]]
    return ['~' + s[1][1]]                # ('not', ('sym', n))

def _conjuncts(s):
    if s[0] == 'and': return _conjuncts(s[1]) + _conjuncts(s[2])
    return [s]

def is_tautology(clause):
    return any(neg(l) in clause for l in clause)

def to_clauses(s):
    clauses = set()
    for part in _conjuncts(to_cnf(s)):
        c = frozenset(_literals(part))
        if not is_tautology(c):
            clauses.add(c)
    return clauses

In [8]:
# Шалгах (Алхам 4)
A, B, C = Sym('A'), Sym('B'), Sym('C')
cl = to_clauses(Iff(B, Or(A, C)))
expected = {frozenset({'~B', 'A', 'C'}), frozenset({'~A', 'B'}), frozenset({'~C', 'B'})}
assert cl == expected and len(cl) == 3
print([sorted(c) for c in cl])
tests = [Iff(P, Q), Imp(And(P, Q), R), Not(Or(Imp(P, Q), Iff(Q, R))),
         Or(And(P, Q), And(Not(P), R)), Iff(Iff(P, Q), R)]
for t in tests:
    c = to_cnf(t)
    same = all(pl_true(t, m) == pl_true(c, m) for m in all_models(symbols(t)))
    print(same, to_str(t))
    assert same

[['B', '~C'], ['A', 'C', '~B'], ['B', '~A']]
True (P <=> Q)
True ((P & Q) => R)
True ~((P => Q) | (Q <=> R))
True ((P & Q) | (~P & R))
True ((P <=> Q) <=> R)


## Алхам 5: Резолюц

In [9]:
class TooSlow(Exception):
    pass

RES_COUNT = 0      # үүсгэсэн resolvent-ийн тоо
RES_LIMIT = 100_000

def resolve(ci, cj):
    result = []
    for lit in ci:
        if neg(lit) in cj:
            new = (ci - {lit}) | (cj - {neg(lit)})
            if not is_tautology(new):
                result.append(frozenset(new))
    return result

def pl_resolution_unsat(clauses):
    """Clause-ийн олонлог unsatisfiable эсэхийг резолюцоор шалгана."""
    global RES_COUNT
    clauses = set(clauses)
    while True:
        new = set()
        lst = list(clauses)
        for i in range(len(lst)):
            for j in range(i + 1, len(lst)):
                for r in resolve(lst[i], lst[j]):
                    RES_COUNT += 1
                    if RES_COUNT > RES_LIMIT:
                        raise TooSlow()
                    if not r:
                        return True           # хоосон clause
                    new.add(r)
        if new <= clauses:
            return False                      # шинэ clause үүссэнгүй
        clauses |= new

def pl_resolution(kb_clauses, alpha):
    """KB |= alpha эсэхийг refutation-аар шалгана."""
    global RES_COUNT
    RES_COUNT = 0
    return pl_resolution_unsat(set(kb_clauses) | to_clauses(Not(alpha)))

In [10]:
# Шалгах (Алхам 5)
assert resolve(frozenset({'A', 'B'}), frozenset({'~A', 'C'})) == [frozenset({'B', 'C'})]
assert resolve(frozenset({'C'}), frozenset({'~C'})) == [frozenset()]
kb5 = {frozenset({'A', 'B'}), frozenset({'~A', 'C'}), frozenset({'~B', 'C'})}
assert pl_resolution(kb5, C) is True
assert pl_resolution(kb5, A) is False
print('резолюц: OK')

резолюц: OK


## Алхам 6: DPLL

In [11]:
DPLL_CALLS = 0

def lit_value(lit, model):
    name = lit.lstrip('~')
    if name not in model: return None
    return (not model[name]) if lit.startswith('~') else model[name]

def clause_status(clause, model):
    """True: хангагдсан, False: бүх литерал худал, None: тодорхойгүй."""
    undecided = False
    for lit in clause:
        v = lit_value(lit, model)
        if v is True: return True
        if v is None: undecided = True
    return None if undecided else False

def find_pure_symbol(symbols_, clauses, model):
    live = [c for c in clauses if clause_status(c, model) is not True]
    for sym in symbols_:
        pos = any(sym in c for c in live)
        ng = any('~' + sym in c for c in live)
        if pos and not ng: return sym, True
        if ng and not pos: return sym, False
    return None

def find_unit_clause(clauses, model):
    for c in clauses:
        if clause_status(c, model) is None:
            free = [l for l in c if lit_value(l, model) is None]
            if len(free) == 1:
                l = free[0]
                return l.lstrip('~'), not l.startswith('~')
    return None

def dpll(clauses, symbols_, model, use_pure=True, use_unit=True):
    global DPLL_CALLS
    DPLL_CALLS += 1
    statuses = [clause_status(c, model) for c in clauses]
    if any(s is False for s in statuses): return False
    if all(s is True for s in statuses): return model
    if use_pure:
        p = find_pure_symbol(symbols_, clauses, model)
        if p:
            sym, val = p
            return dpll(clauses, [x for x in symbols_ if x != sym], {**model, sym: val}, use_pure, use_unit)
    if use_unit:
        u = find_unit_clause(clauses, model)
        if u:
            sym, val = u
            return dpll(clauses, [x for x in symbols_ if x != sym], {**model, sym: val}, use_pure, use_unit)
    first, rest = symbols_[0], symbols_[1:]
    for val in (True, False):
        r = dpll(clauses, rest, {**model, first: val}, use_pure, use_unit)
        if r is not False: return r
    return False

def dpll_satisfiable(clauses, use_pure=True, use_unit=True):
    global DPLL_CALLS
    DPLL_CALLS = 0
    clauses = list(clauses)
    syms = sorted({l.lstrip('~') for c in clauses for l in c})
    return dpll(clauses, syms, {}, use_pure, use_unit)

In [12]:
# Шалгах (Алхам 6)
f = lambda *x: frozenset(x)
cl6 = [f('A'), f('~A', 'B'), f('~B', 'C'), f('~C', 'D', 'E'), f('~D', '~E')]
m = dpll_satisfiable(cl6)
assert m and m['A'] and m['B'] and m['C'] and (m['D'] != m['E']), m
print('satisfiable загвар:', m)
unsat = [f('A', 'B'), f('~A', 'B'), f('A', '~B'), f('~A', '~B')]
assert dpll_satisfiable(unsat) is False
# KB |= C  <=>  CNF(KB & ~C) unsatisfiable
assert dpll_satisfiable(set(kb5) | to_clauses(Not(C))) is False
assert dpll_satisfiable(set(kb5) | to_clauses(Not(A))) is not False
print('DPLL: OK')

satisfiable загвар: {'A': True, 'B': True, 'C': True, 'D': True, 'E': False}
DPLL: OK


## Алхам 7: Wumpus-ийн мэдлэгийн сан

In [13]:
P11, P12, P21, P22, P31 = (Sym(x) for x in ['P11', 'P12', 'P21', 'P22', 'P31'])
B11, B21 = Sym('B11'), Sym('B21')
rules = [Not(P11),
         Iff(B11, Or(P12, P21)),
         Iff(B21, Or(Or(P11, P22), P31)),
         Not(B11),
         B21]
wumpus_kb = conj(rules)
wumpus_clauses = to_clauses(wumpus_kb)
queries = {'~P12': Not(P12), '~P21': Not(P21), '~P22': Not(P22),
           '~P31': Not(P31), 'P22 v P31': Or(P22, P31)}

def three_ways(kb_sentence, kb_clauses, alpha):
    tt, n_tt = tt_entails(kb_sentence, alpha)
    rs = pl_resolution(kb_clauses, alpha); n_rs = RES_COUNT
    dp = dpll_satisfiable(set(kb_clauses) | to_clauses(Not(alpha))) is False; n_dp = DPLL_CALLS
    return (tt, n_tt), (rs, n_rs), (dp, n_dp)

In [14]:
n_models = sum(1 for m in all_models(symbols(wumpus_kb)) if pl_true(wumpus_kb, m))
print('KB-ийг хангах загварын тоо:', n_models); assert n_models == 3
expected7 = {'~P12': True, '~P21': True, '~P22': False, '~P31': False, 'P22 v P31': True}
wumpus_results = {}
for name, alpha in queries.items():
    (tt, ntt), (rs, nrs), (dp, ndp) = three_ways(wumpus_kb, wumpus_clauses, alpha)
    wumpus_results[name] = (ntt, nrs, ndp)
    print(f'{name:10} tt={tt} ({ntt:3}) | res={rs} ({nrs:4}) | dpll={dp} ({ndp:2})')
    assert tt == rs == dp == expected7[name]

KB-ийг хангах загварын тоо: 3
~P12       tt=True (128) | res=True (  13) | dpll=True ( 4)
~P21       tt=True (128) | res=True (  13) | dpll=True ( 4)
~P22       tt=False ( 93) | res=False ( 127) | dpll=False ( 7)
~P31       tt=False ( 93) | res=False ( 127) | dpll=False ( 7)
P22 v P31  tt=True (128) | res=True ( 282) | dpll=True ( 8)


## Алхам 8: Харьцуулалт

In [15]:
def random_3sat(n, m, seed):
    random.seed(seed)
    clauses = []
    for _ in range(m):
        vs = random.sample(range(1, n + 1), 3)
        clauses.append(frozenset(('' if random.random() < 0.5 else '~') + f'X{v}' for v in vs))
    return clauses

TT_LIMIT = 200_000

def tt_satisfiable(clauses):
    """Үнэний хүснэгтээр satisfiability. (үр дүн, шалгасан загвар) эсвэл (None, ...)."""
    syms = sorted({l.lstrip('~') for c in clauses for l in c})
    checked = 0
    for model in all_models(syms):
        checked += 1
        if checked > TT_LIMIT: return None, checked
        if all(clause_status(c, model) is True for c in clauses):
            return True, checked
    return False, checked

def compare_3sat(n, seed=1):
    cls = random_3sat(n, 4 * n, seed)
    out = {}
    sat_tt, n_tt = tt_satisfiable(cls)
    out['tt'] = 'хэт удаан' if sat_tt is None else n_tt
    global RES_COUNT
    RES_COUNT = 0
    try:
        unsat = pl_resolution_unsat(set(cls)); out['res'] = RES_COUNT
    except TooSlow:
        unsat = None; out['res'] = 'хэт удаан'
    sat_dp = dpll_satisfiable(cls) is not False
    out['dpll'] = DPLL_CALLS
    out['sat'] = sat_dp
    return out

In [16]:
table = {}
for name, (ntt, nrs, ndp) in wumpus_results.items():
    if name in ('~P12', '~P22', 'P22 v P31'):
        table['Wumpus: ' + name] = {'tt': ntt, 'res': nrs, 'dpll': ndp}
for n in (10, 15, 20):
    table[f'3-SAT, n = {n}'] = compare_3sat(n)
print(f"{'Бодлого':22}{'Үнэний хүснэгт':>16}{'Резолюц':>12}{'DPLL':>8}  satisfiable")
for k, v in table.items():
    print(f"{k:22}{str(v['tt']):>16}{str(v['res']):>12}{str(v['dpll']):>8}  {v.get('sat', '')}")

Бодлого                 Үнэний хүснэгт     Резолюц    DPLL  satisfiable
Wumpus: ~P12                       128          13       4  
Wumpus: ~P22                        93         127       7  
Wumpus: P22 v P31                  128         282       8  
3-SAT, n = 10                      368   хэт удаан      30  True
3-SAT, n = 15                    14770   хэт удаан      99  True
3-SAT, n = 20                хэт удаан   хэт удаан     103  True


## Нэмэлт даалгавар 1: Рыцарь ба луйварчин

In [17]:
KA, KB, KC = Sym('KA'), Sym('KB'), Sym('KC')
knight_kb = conj([Iff(KA, Not(KB)),            # A: "B бол луйварчин"
                  Iff(KB, Iff(KA, KC)),        # B: "A ба C ижил төрлийн"
                  Iff(KC, KB)])                # C: "B бол рыцарь"
sols = [m for m in all_models(symbols(knight_kb)) if pl_true(knight_kb, m)]
print('Хангах загвар:', sols); assert len(sols) == 1
for k, v in sols[0].items():
    print(k, '-> рыцарь' if v else '-> луйварчин')
print('KB |= KA :', tt_entails(knight_kb, KA))
print('KB |= ~KB:', tt_entails(knight_kb, Not(KB)))
print('KB |= ~KC:', tt_entails(knight_kb, Not(KC)))

Хангах загвар: [{'KA': True, 'KB': False, 'KC': False}]
KA -> рыцарь
KB -> луйварчин
KC -> луйварчин
KB |= KA : (True, 8)
KB |= ~KB: (True, 8)
KB |= ~KC: (True, 8)


## Нэмэлт даалгавар 2: DPLL-ийн unit / pure дүрмийг унтраах

In [18]:
problems = {}
for name in ('~P12', '~P22', 'P22 v P31'):
    problems['Wumpus: ' + name] = set(wumpus_clauses) | to_clauses(Not(queries[name]))
for n in (10, 15, 20):
    problems[f'3-SAT, n = {n}'] = random_3sat(n, 4 * n, 1)
ablation = {}
print(f"{'Бодлого':22}{'Бүгд':>8}{'Pure-гүй':>10}{'Unit-гүй':>10}{'Хоёул үгүй':>12}")
for name, cls in problems.items():
    row = []
    for up, uu in [(True, True), (False, True), (True, False), (False, False)]:
        dpll_satisfiable(cls, use_pure=up, use_unit=uu); row.append(DPLL_CALLS)
    ablation[name] = row
    print(f'{name:22}{row[0]:>8}{row[1]:>10}{row[2]:>10}{row[3]:>12}')

Бодлого                   Бүгд  Pure-гүй  Unit-гүй  Хоёул үгүй
Wumpus: ~P12                 4         3        10           9
Wumpus: ~P22                 7         7         8          11
Wumpus: P22 v P31            8         8        13          15
3-SAT, n = 10               30        30       122         168
3-SAT, n = 15               99        92       441         858


3-SAT, n = 20              103        96      2053        2449
